In [ ]:
import pandas as pd
import altair as alt

In [ ]:
lib_a_oligos =pd.read_csv('/Users/ivan/Documents/GitHub/BARD1_SGE_analysis/Data/extra_data/20260603_BARD1_X4A_IDR_LibA.csv')
lib_b_oligos=pd.read_csv('/Users/ivan/Documents/GitHub/BARD1_SGE_analysis/Data/extra_data/20260603_BARD1_X4A_IDR_LibB.csv')

lib_a_lfcs=pd.read_csv('/Users/ivan/Documents/GitHub/BARD1_SGE_analysis/Data/extra_data/20260812_BARD1_X4A_IDR_LibA_LFCs.tsv', sep='\t')
lib_b_lfcs=pd.read_csv('/Users/ivan/Documents/GitHub/BARD1_SGE_analysis/Data/extra_data/20260812_BARD1_X4A_IDR_LibB_LFCs.tsv', sep='\t')

# Analyze Library A

In [ ]:
lib_a_merged = pd.merge(lib_a_lfcs, lib_a_oligos, left_on='oligo_name', right_on='seq_name', how='inner')

lib_a_merged

In [ ]:
## Validate that name-matched rows also agree on sequence.
# lib_a_oligos['seq'] is the full designed oligo (AMP_F + edited region + AMP_R,
# sense strand). lib_a_lfcs['oligo_seq'] is only the edited core region, read off
# the antisense strand. So to confirm the merge isn't just matching names while
# silently pairing unrelated sequences, trim the primers off 'seq' and compare
# to the reverse complement of 'oligo_seq'. See SGE_BARD1_X4_IDR_libs.ipynb for
# BARD1_X4a_AMP_F / BARD1_X4a_AMP_R and how the oligos were built.

BARD1_X4a_AMP_F = 'CCATGTGGGAGCAATAAATTTC'
BARD1_X4a_AMP_R = 'CCCTCGAAGTAAGAAAGTCAG'

def revcomp(seq):
    comp = str.maketrans('ACGTacgt', 'TGCAtgca')
    return seq.translate(comp)[::-1]

def core_region(seq):
    seq = seq.upper()
    assert seq.startswith(BARD1_X4a_AMP_F), f'missing AMP_F: {seq[:30]}'
    assert seq.endswith(BARD1_X4a_AMP_R), f'missing AMP_R: {seq[-30:]}'
    return seq[len(BARD1_X4a_AMP_F):len(seq) - len(BARD1_X4a_AMP_R)]

seq_matches = lib_a_merged['seq'].apply(core_region) == lib_a_merged['oligo_seq'].apply(revcomp).str.upper()

assert seq_matches.all(), f'{(~seq_matches).sum()} of {len(lib_a_merged)} merged rows have mismatched sequences'
print(f'All {len(lib_a_merged)} merged rows agree on sequence (name match backed by sequence match).')

lib_a_final=lib_a_merged[['oligo_name', 'oligo_seq', 'var_name', 'D13_log2_median']].copy()

lib_a_final